# 📗 판다스 기초 — 결측치·이상치·형변환·문자열·파생 변수

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

> 📚 **pandas 공식 문서**: https://pandas.pydata.org/docs/ · 입문 가이드 [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) · API 검색 [API reference](https://pandas.pydata.org/docs/reference/index.html)

## 오늘의 목표
- [ ] **결측치(NaN)** 를 찾아내고(`isna`) 지우거나(`dropna`) 채운다(`fillna`).
- [ ] **이상치**(999 같은 튀는 값)를 탐지하고 걸러낸다.
- [ ] **자료형을 변환**한다 — 문자열 가격 → 정수, 문자열 날짜 → datetime.
- [ ] **문자열 열**을 `.str` 로 다듬는다 (공백 제거·치환·검색).
- [ ] 기존 열로 **새 열(파생 변수)** 을 만든다 — `apply`·`map`·`np.where`.

## ⏪ 복습 — 지난 시간엔 데이터를 훑고 골라냈다
앞서 우리는 표를 **훑어보고(`head`·`info`·`describe`)**, **골라내고(열 선택·`loc`·`iloc`·불리언 필터)**, **정렬**했습니다.
그 과정에서 눈에 걸린 것들이 있었죠.
- 수량에 **999** 같은 말도 안 되는 값, **비어 있는(NaN)** 칸
- **가격이 문자열** — `"5,000"` 처럼 콤마가 섞여 계산이 안 됨
- **음료 이름에 앞뒤 공백** — `"  레모네이드 "` 와 `"레모네이드"` 가 다른 값 취급

지저분한 데이터로는 올바른 분석이 나올 수 없습니다. 이어서 이 데이터를 **말끔히 정제**하고 **새 정보를 만들어** 봅니다. 🚀

---
# 1. 결측치 처리 — 비어 있는 칸 다루기

## 왜 필요할까요?
실무 데이터엔 **비어 있는 칸**이 흔합니다. 판다스는 이를 **`NaN`**(Not a Number)으로 표시해요.
결측치를 그대로 두면 평균·합계가 틀어지므로, **찾아서 → 지우거나 → 채워야** 합니다.

| 도구 | 하는 일 |
|---|---|
| `df['열'].isna()` | 각 칸이 비었는지 True/False |
| `df['열'].isna().sum()` | 결측이 몇 개인지 세기 |
| `df['열'].notna()` | 비어 있지 **않은** 칸 |
| `df.dropna(subset=['열'])` | 그 열이 빈 **행을 삭제** |
| `df['열'].fillna(값)` | 빈 칸을 **값으로 채움** |

> **지울까 채울까?** — 데이터가 넉넉하면 지우고(`dropna`), 아까우면 대표값(중앙값 등)으로 채웁니다(`fillna`).

In [ ]:
# [제공 코드] 오후에도 같은 라이브러리와 데이터를 씁니다.
import pandas as pd
import numpy as np
df = pd.read_csv("data/cafe_sales.csv")
print("데이터 크기:", df.shape)

In [ ]:
# 결측치 찾기 — 어느 열에 몇 개나 비어 있나?
print("--- 열별 결측 개수 ---")
print(df.isna().sum())          # '수량' 열에 5개
print("수량 결측 개수:", df['수량'].isna().sum(), "건")
# notna() 는 isna() 의 반대 — 값이 '있는' 칸을 셀 때 쓴다
print("수량 값이 있는 칸:", df['수량'].notna().sum(), "건")

In [ ]:
# 결측 행 삭제 vs 채우기 (원본은 그대로 두고 결과만 확인)
dropped = df.dropna(subset=['수량'])
print("결측 행 삭제 후:", dropped.shape[0], "행")          # 55행 (60-5)

filled_zero = df['수량'].fillna(0)
print("0으로 채운 뒤 결측:", filled_zero.isna().sum(), "건")

# 대표값으로 채우기 — 평균은 이상치(999)에 크게 흔들리니 '중앙값'이 안전합니다.
print("수량 평균:", round(df['수량'].mean(), 2), "(999 때문에 부풀었음)")   # 39.09
print("수량 중앙값:", df['수량'].median(), "(이상치에 안 흔들림)")            # 3.0
filled_median = df['수량'].fillna(df['수량'].median())
print("중앙값으로 채운 뒤 결측:", filled_median.isna().sum(), "건")

빈 칸을 **바로 앞/뒤 값**으로 채울 수도 있습니다 — 순서가 있는 데이터(시간·정렬된 표)에서 자연스럽습니다.

| 코드 | 뜻 |
|---|---|
| `df['수량'].ffill()` | 바로 **앞** 값으로 채움 (forward fill) |
| `df['수량'].bfill()` | 바로 **뒤** 값으로 채움 (backward fill) |

In [ ]:
# 앞/뒤 값으로 결측 채우기
cafe = pd.read_csv('data/cafe_sales.csv')
print('원래 결측:', cafe['수량'].isna().sum())
print('ffill 후 결측:', cafe['수량'].ffill().isna().sum())
print('bfill 후 결측:', cafe['수량'].bfill().isna().sum())

### 🖐️ 함께 따라하기 — 빈 칸 메우기 (동네 서점 데이터)
데모는 카페였죠? 이제 **다른 가게(동네 서점) 데이터**를 직접 불러와, 판매부수 열의 결측을 찾고 지우기·채우기를 모두 해 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# ※ 데모는 '카페' 데이터였죠. 이번엔 다른 가게 데이터(동네 서점)로 정제를 연습합니다.
# 1) pd.read_csv 로 data/bookstore_sales.csv 를 읽어 books 에 담는다
# 2) books['판매부수'].isna().sum() 으로 결측 개수를 출력한다
# 3) dropna(subset=['판매부수']) 로 결측 행을 지운 표의 행 수를 출력한다
# 4) '판매부수'의 중앙값을 구하고, fillna 로 결측을 그 값으로 채워 결측이 0개가 됐는지 확인한다

### ✅ 바로 확인 퀴즈
**1.** 비어 있는 칸을 판다스는 무엇으로 표시하나요?

<details>
<summary>정답 보기</summary>

**`NaN`**(Not a Number)으로 표시합니다. `isna()` 가 True 를 돌려주는 칸이에요.

</details>

**2.** 결측을 채울 때 평균보다 **중앙값**이 안전한 경우는 언제일까요?

<details>
<summary>정답 보기</summary>

**이상치(극단값)가 섞여 있을 때**입니다. 여기 수량 평균은 999 때문에 39.09로 부풀었지만 중앙값은 3.0으로 멀쩡하죠. 중앙값은 극단값에 잘 흔들리지 않습니다.

</details>

---
# 2. 이상치 처리 — 튀는 값 걸러내기

## 왜 필요할까요?
**이상치(outlier)** 는 정상 범위를 크게 벗어난 값입니다. 여기선 수량 **999** 가 그렇죠(커피 999잔?).
이상치 하나가 평균을 통째로 왜곡합니다(수량 평균 39.09 vs 중앙값 3.0).

탐지·처리 방법(오늘 범위 안에서):

| 방법 | 문법 | 설명 |
|---|---|---|
| 통계로 감지 | `df['열'].describe()` | 최댓값이 유독 크면 의심 |
| 조건으로 제거 | `df[df['열'] != 999]` | 불리언 필터로 걸러내기 |
| 상한 자르기 | `df['열'].clip(upper=10)` | 큰 값을 상한으로 눌러줌 |
| 분위수 경계 | `df['열'].quantile(0.95)` | 상위 5% 경계값 찾기 |

In [ ]:
# 통계로 이상치 냄새 맡기 — 평균과 최댓값이 너무 벌어져 있다
print(df['수량'].describe())
print("평균:", round(df['수량'].mean(), 2), "/ 중앙값:", df['수량'].median(), "/ 최대:", df['수량'].max())

In [ ]:
# 조건 필터로 999 이상치 제거 (오전에 배운 불리언 필터!)
no_outlier = df[df['수량'] != 999]
print("999 제거 후 평균:", round(no_outlier['수량'].mean(), 2))   # 2.87 로 정상화
print("제거된 이상치 건수:", (df['수량'] == 999).sum(), "건")     # 2건

In [ ]:
# clip — 상한을 넘는 값을 상한으로 눌러 담기 / quantile — 경계값 찾기
clipped = df['수량'].clip(upper=10)
print("clip(상한 10) 후 최댓값:", clipped.max())               # 10.0
print("상위 5% 경계(0.95 분위):", df['수량'].quantile(0.95))   # 5.0

### 🖐️ 함께 따라하기 — 이상치 솎아내기
판매부수의 이상치를 통계로 확인하고, 조건 필터로 제거해 평균이 정상으로 돌아오는 걸 눈으로 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) books['판매부수'].max() 로 최댓값을 출력해 이상치(999)를 확인한다
# 2) 조건 필터 books[books['판매부수'] != 999] 로 이상치를 뺀 표를 clean 에 담는다
# 3) 제거 전/후의 '판매부수' 평균을 각각 round(...,2) 로 출력해 비교한다
# 4) clip(upper=10) 으로 큰 값을 10으로 누른 시리즈의 최댓값을 출력한다

### ✅ 바로 확인 퀴즈
**1.** 수량 999를 조건 필터로 빼려면 어떻게 쓸까요?

<details>
<summary>정답 보기</summary>

`df[df['수량'] != 999]` — 앞서 배운 불리언 필터를 그대로 씁니다. `!=` 는 '같지 않다'예요.

</details>

**2.** 이상치를 지우지 않고 **상한선으로 눌러** 담고 싶습니다. 어떤 메서드를 쓸까요?

<details>
<summary>정답 보기</summary>

`clip(upper=상한)` 입니다. 예: `df['수량'].clip(upper=10)` 은 10보다 큰 값을 모두 10으로 바꿉니다(아래쪽은 `lower=`).

</details>

---
# 3. 자료형 변환 — 문자열을 숫자·날짜로

## 왜 필요할까요?
`info()` 에서 봤듯 **가격이 문자열(str)** 이었습니다. `"5,000"` 처럼 **콤마**가 섞인 값이 하나라도 있으면 판다스는 그 열 전체를 문자열로 읽어요. 문자열끼리는 곱셈·평균이 안 됩니다.
**주문일**도 문자열이라, 요일을 뽑으려면 **날짜형(datetime)** 으로 바꿔야 합니다.

| 문법 | 하는 일 |
|---|---|
| `s.astype(int)` | 자료형 강제 변환 (깨끗한 값일 때) |
| `s.str.replace(',', '')` | 콤마 같은 문자를 먼저 제거 |
| `pd.to_numeric(s, errors='coerce')` | 숫자로 못 바꾸는 값은 NaN 처리 |
| `pd.to_datetime(s)` | 문자열을 날짜형으로 |

> **순서가 중요** — 가격은 콤마를 **먼저 지우고**(`.str.replace`) 그다음에 정수로 바꿉니다.

In [ ]:
# 지금 자료형 확인 — 가격이 문자열(str)이다
# (판다스는 문자열 열을 dtypes 에서 'object' 로 표시합니다 — object = 문자열이라는 뜻)
print(df.dtypes)
print("가격 값 예시:", df['가격'].unique()[:6])   # '5,000' 같은 콤마 값이 섞여 있음

In [ ]:
# 가격: 콤마 제거 -> 정수 변환
price_int = df['가격'].str.replace(',', '').astype(int)
print("변환 후 자료형:", price_int.dtype)          # int64
print("가격 합계:", price_int.sum(), "원")         # 273900
print("가격 평균:", round(price_int.mean(), 1), "원")   # 4565.0

# 콤마를 안 지우고 그냥 숫자변환하면? to_numeric(errors='coerce')는 실패 값을 NaN 으로.
coerced = pd.to_numeric(df['가격'], errors='coerce')
print("콤마를 안 지우면 NaN 이 되는 값:", coerced.isna().sum(), "건")   # 12건

In [ ]:
# 주문일: 문자열 -> 날짜형(datetime)
order_date = pd.to_datetime(df['주문일'])
print("변환 후 자료형:", order_date.dtype)
print("가장 이른 주문:", order_date.min().date())   # 2026-06-01
print("가장 늦은 주문:", order_date.max().date())   # 2026-06-28

In [ ]:
# .dt 로 날짜 정보 뽑기 — 시각(시:분)이 있으면 '몇 시 주문'까지 분석할 수 있다
sample = pd.to_datetime(pd.Series(["2026-06-27 14:30", "2026-06-28 09:05"]))
print("연도(year):", list(sample.dt.year))    # [2026, 2026]
print("일(day):   ", list(sample.dt.day))     # [27, 28]
print("시(hour):  ", list(sample.dt.hour))    # [14, 9]
# (카페 데이터의 '주문일'은 날짜만 있지만, 과제의 shop_orders '주문일시'엔 시각이 들어 있어
#  이렇게 .dt.hour 로 시간대별 주문을 분석할 수 있어요.)

### 🖐️ 함께 따라하기 — 자료형 바로잡기
문자열 정가를 정수로, 문자열 날짜를 datetime 으로 바꿔 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) books['정가'] 의 콤마를 .str.replace(',', '') 로 지운다
# 2) 이어서 .astype(int) 로 정수로 바꿔 변수 price_num 에 담는다
# 3) price_num 의 평균을 round(...,1) 로 출력한다
# 4) pd.to_datetime 으로 '판매일'을 날짜형으로 바꿔 최솟값·최댓값 날짜를 출력한다

### ✅ 바로 확인 퀴즈
**1.** `df['가격'].astype(int)` 이 바로 에러를 냅니다. 왜일까요?

<details>
<summary>정답 보기</summary>

`"5,000"` 처럼 **콤마가 든 문자열**은 정수로 못 바꾸기 때문입니다. `.str.replace(',', '')` 로 콤마를 **먼저 지운 뒤** `astype(int)` 해야 해요.

</details>

**2.** 문자열 `"2026-06-27"` 을 날짜형으로 바꾸는 함수는?

<details>
<summary>정답 보기</summary>

`pd.to_datetime(...)` 입니다. 날짜형으로 바꾸면 `.dt.year`·`.dt.dayofweek` 처럼 날짜 정보를 뽑을 수 있어요.

</details>

---
# 4. 문자열 처리 — .str 로 텍스트 다듬기

## 왜 필요할까요?
음료 이름에 `"  레모네이드 "` 처럼 **앞뒤 공백**이 섞여 있었죠. 공백이 있으면 같은 음료가 **다른 값**으로 취급됩니다.
판다스는 문자열 열에 **`.str`** 을 붙이면, 파이썬 문자열 메서드를 **열 전체에 한 번에** 적용해 줍니다.

| 문법 | 하는 일 |
|---|---|
| `s.str.strip()` | 앞뒤 공백 제거 |
| `s.str.replace('a','b')` | 글자 치환 |
| `s.str.contains('라떼')` | 특정 글자 포함 여부(True/False) |
| `s.str.lower()` / `s.str.upper()` | 소문자/대문자 |
| `s.str.len()` | 글자 수 |
| `s.str.split('-')` | 구분자로 쪼개기 |

In [ ]:
# 공백 제거 전/후 — 공백 때문에 음료 종류가 부풀어 보인다
print("공백 정리 전 음료 종류 수:", df['음료'].nunique(), "개")
clean_drink = df['음료'].str.strip()
print("공백 정리 후 음료 종류 수:", clean_drink.nunique(), "개")   # 10개
print("정리 전 예시:", repr(df['음료'].iloc[0]), "→", repr(clean_drink.iloc[0]))

In [ ]:
# contains — 특정 단어가 든 음료 찾기 / len — 이름 글자 수
print("이름에 '라떼'가 든 주문:", clean_drink.str.contains('라떼').sum(), "건")   # 19건
print("이름에 '아메'가 든 주문:", clean_drink.str.contains('아메').sum(), "건")   # 7건
print("음료 이름 글자 수(앞 3개):", list(clean_drink.str.len().head(3)))
# 참고: 영문 텍스트라면 .str.lower() / .str.upper() 로 대소문자를 통일할 수 있다
#       (예: 'Python' 과 'python' 을 같게 맞추기) — 오후 따라하기에서 직접 써 본다

In [ ]:
# split — 구분자로 쪼개기 (주문일에서 연/월/일 분리)
parts = df['주문일'].str.split('-')
print("'2026-06-27' → ", parts.iloc[0])            # ['2026', '06', '27']
# str[1] 로 '월' 부분만 뽑기
month = df['주문일'].str.split('-').str[1]
print("월만 뽑기(앞 3개):", list(month.head(3)))

### 문자열 이어 붙이기 — Series `+` 결합

문자열 열은 파이썬 문자열처럼 `+` 로 이어 붙일 수 있습니다. 숫자 열을 섞을 때는 `astype(str)` 로 **먼저 문자열로 바꿔야** `문자열 + 숫자` 에러가 나지 않습니다.

In [ ]:
# 음료 이름과 카테고리를 한 문구로 잇기 (숫자는 문자열로 변환 후 결합)
cafe = pd.read_csv('data/cafe_sales.csv')
cafe['음료'] = cafe['음료'].str.strip()
cafe['수량'] = cafe['수량'].fillna(0)
label = cafe['음료'] + ' / ' + cafe['카테고리']
qty_label = cafe['음료'] + ' x' + cafe['수량'].astype(int).astype(str)
print(label.iloc[0])
print(qty_label.iloc[0])

### 🖐️ 함께 따라하기 — 텍스트 청소
도서명의 공백을 지우고, 특정 단어가 든 도서를 세어 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) books['도서명'] 의 앞뒤 공백을 .str.strip() 으로 지워 title 에 담는다
# 2) title.nunique() 로 실제 도서 종류 수를 출력한다
# 3) title.str.contains('python') 으로 'python'이 든 도서 수를 출력한다
# 4) title.str.upper() 로 도서명을 모두 대문자로 바꿔 앞 3개를 출력한다 (영문 'python' -> 'PYTHON')
# 5) title.str.len() 으로 각 도서명 글자 수를 구해 앞 5개를 출력한다
# 6) title 과 books['분야'] 를 ' / ' 로 이어 붙여(문자열 +) 첫 값을 출력한다 (예: 'python 실전 / 자기계발')

### ✅ 바로 확인 퀴즈
**1.** `"  레모네이드 "` 의 앞뒤 공백을 없애는 문자열 도구는?

<details>
<summary>정답 보기</summary>

`.str.strip()` 입니다. 열 전체에 적용하려면 `df['음료'].str.strip()` 처럼 **`.str`** 을 붙여요.

</details>

**2.** 음료 이름에 '라떼'가 들어간 행만 고르고 싶습니다. `.str` 무엇을 쓸까요?

<details>
<summary>정답 보기</summary>

`.str.contains('라떼')` 로 True/False 를 만든 뒤 `df[df['음료'].str.strip().str.contains('라떼')]` 처럼 불리언 필터에 넣습니다.

</details>

---
# 5. 파생 변수 — 기존 열로 새 열 만들기

## 왜 필요할까요?
가진 열을 조합하면 **새로운 정보**를 만들 수 있습니다. 예: 가격 × 수량 = **총액**, 날짜 → **요일**.
이렇게 만든 열을 **파생 변수**라고 해요. 새 열은 그냥 `df['새열'] = ...` 로 대입하면 생깁니다.

| 방법 | 언제 | 예시 |
|---|---|---|
| 열끼리 연산 | 사칙연산으로 충분할 때 | `df['총액'] = df['가격'] * df['수량']` |
| `apply(함수)` | 값마다 **직접 만든 함수**를 적용 | 가격 → '고가'/'보통' 등급 |
| `map(딕셔너리)` | 값을 **1:1 대응표**로 치환 | `'Y'`→`'회원'` |
| `np.where(조건, A, B)` | 조건에 따라 **둘 중 하나** | 수량 4 이상이면 '대량' |

> 파생 변수를 만들기 전에 먼저 **정제**(공백·콤마·이상치·결측)를 끝내야 계산이 올바릅니다. 아래는 앞서 배운 걸 모두 이어 붙인 흐름이에요.

In [ ]:
# 먼저 정제 — 지금까지 배운 것을 work 사본에 모아 적용
# .copy() 로 사본을 만드는 이유: 원본 df 오염을 막고, 사본 여부가 애매할 때 뜨는
#   SettingWithCopyWarning 경고도 피한다. 원본은 두고 사본에만 정제/파생을 쌓자.
work = df.copy()
work['음료'] = work['음료'].str.strip()                         # 공백 제거
work['가격'] = work['가격'].str.replace(',', '').astype(int)    # 콤마 제거 -> 정수
work.loc[work['수량'] == 999, '수량'] = np.nan                  # 이상치를 결측 처리
work['수량'] = work['수량'].fillna(work['수량'].median()).astype(int)   # 중앙값으로 채우고 정수화
print("정제 후 남은 결측 총합:", work.isna().sum().sum(), "건")   # 0
print("정제 후 수량 최댓값:", work['수량'].max())                 # 5

In [ ]:
# 열끼리 연산으로 '총액' 파생 변수 만들기
work['총액'] = work['가격'] * work['수량']
print("총 매출:", work['총액'].sum(), "원")            # 791000
print("주문당 평균 총액:", round(work['총액'].mean(), 1), "원")   # 13183.3
work[['음료', '가격', '수량', '총액']].head(3)

In [ ]:
# apply — 값마다 내가 만든 함수를 적용해 '가격등급' 만들기
def grade_price(price):
    return '고가' if price >= 5000 else '보통'

work['가격등급'] = work['가격'].apply(grade_price)
# value_counts() — 범주형 열에서 값별 개수를 세어 준다(분포 한눈에 보기). 아래에서 자주 쓴다.
print(work['가격등급'].value_counts())   # 보통 33, 고가 27

In [ ]:
# map — 대응표(딕셔너리)로 값 치환 / 날짜에서 요일 뽑기
member_label = {'Y': '회원', 'N': '비회원'}
work['회원구분'] = work['회원여부'].map(member_label)
print(work['회원구분'].value_counts())   # 회원 33, 비회원 27

# 요일: 날짜형으로 바꾼 뒤 요일 번호(0=월)를 한글로 매핑
week_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
order_date = pd.to_datetime(work['주문일'])
work['요일'] = order_date.dt.dayofweek.map(week_map)
print(work['요일'].value_counts())

In [ ]:
# np.where — 조건에 따라 둘 중 하나로 '대량주문' 플래그
work['대량주문'] = np.where(work['수량'] >= 4, '대량', '일반')
print(work['대량주문'].value_counts())   # 일반 40, 대량 20

### 🖐️ 함께 따라하기 — 새 정보 만들기
서점 데이터를 정제한 사본(`bwork = books.copy()`)에 총액·고액여부·회원구분·판매등급 파생 변수를 직접 만들어 봅시다. (아래 셀에서 사본을 만든 뒤 진행합니다.)

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 먼저 이 연습용 사본을 간단히 정제합니다 (앞에서 배운 것):
#   bwork = books.copy() -> 도서명 strip -> 정가 콤마 제거 후 int -> 판매부수 결측은 중앙값으로 채워 int
# 그 다음 파생 변수를 만듭니다:
# 1) '총액' = 정가 * 판매부수 (열끼리 연산)
# 2) np.where 로 총액이 50000 이상이면 '고액' 아니면 '일반'인 '고액여부' 열을 만들고 value_counts 출력
# 3) 적립회원 'Y'/'N' 을 map 으로 '회원'/'비회원' 으로 바꾼 '회원구분' 열을 만든다
# 4) apply 로 판매부수가 3 이상이면 '많음' 아니면 '적음'인 '판매등급' 열을 만들어 value_counts 출력

### ✅ 바로 확인 퀴즈
**1.** 가격 × 수량으로 '총액' 열을 새로 만들려면?

<details>
<summary>정답 보기</summary>

`df['총액'] = df['가격'] * df['수량']`. 없던 열 이름에 대입하면 새 열이 생깁니다. 단, 두 열이 **모두 숫자형**이어야 곱셈이 됩니다.

</details>

**2.** `'Y'`→`'회원'`, `'N'`→`'비회원'` 처럼 **1:1 대응표**로 값을 바꾸는 도구는 `apply` 와 `map` 중 무엇이 어울릴까요?

<details>
<summary>정답 보기</summary>

**`map`** 입니다. 딕셔너리 `{'Y':'회원','N':'비회원'}` 를 넘기면 값을 표대로 치환해요. 복잡한 계산이 필요하면 `apply(함수)` 를 씁니다.

</details>

---
## 🚀 응용 클론코딩 — 지저분한 원본을 분석용 데이터로

앞서 배운 정제·파생을 **한 파이프라인**으로 이어, 서점 원본을 분석 가능한 깨끗한 표로 바꿔 봅시다.

**단계**: ① 도서명 공백 제거 → ② 정가 콤마 제거 후 정수 → ③ 판매부수 이상치(999) 결측 처리 후 중앙값으로 채움 → ④ 판매일 날짜형 + 요일 파생 → ⑤ 총액 파생.

> 여기서는 999를 **중앙값으로 대치**하지만, 실무에선 값 자체를 의심해 **원자료를 확인하거나 그 행을 삭제**하는 선택지도 함께 고려합니다. 정답이 하나가 아니에요.

정제가 끝나면 판매부수 결측은 0건이 되고, 총 매출과 최대 총액을 출력해 확인합니다.

<img src="images/cleaning_pipeline.png" alt="데이터 정제 파이프라인" width="820">

In [ ]:
# 🖐️ 함께 따라하기 — 미니 정제 파이프라인 (아래 순서대로 직접 작성해 보세요)
# 1) books.copy() 로 사본 btidy 를 만든다
# 2) btidy['도서명'] 의 앞뒤 공백을 strip 으로 제거
# 3) btidy['정가'] 의 콤마를 지우고 astype(int) 로 정수화
# 4) 판매부수 999 를 np.nan 으로 바꾼 뒤(loc 사용) fillna(중앙값).astype(int)
# 5) btidy['판매일'] 을 to_datetime 으로 바꾸고, dayofweek 를 week_map 으로 매핑해 btidy['요일'] 생성
# 6) btidy['총액'] = 정가 * 판매부수 로 파생하고, 총 매출과 최대 총액을 출력

---
## 정제 결과 저장하기 — `to_csv`

정제·파생을 마친 표는 **파일로 저장해 다음 분석에서 다시 씁니다**. `to_csv` 로 CSV 파일을 만들어요. 저장할 폴더가 없으면 에러가 나므로 `os.makedirs(..., exist_ok=True)` 로 먼저 만들어 둡니다.

| 옵션 | 뜻 |
|---|---|
| `index=False` | 맨 앞 **행 번호(인덱스)를 파일에 쓰지 않음** (안 그러면 이름 없는 첫 열이 하나 더 생겨요) |
| `encoding="utf-8-sig"` | 한글이 **엑셀에서 안 깨지게** (BOM 붙은 UTF-8) |

In [ ]:
# 정제한 표(work)를 CSV 로 저장 — 폴더가 없으면 먼저 만든다
import os
os.makedirs("output", exist_ok=True)
work.to_csv("output/정제결과.csv", index=False, encoding="utf-8-sig")
print("저장 완료: output/정제결과.csv")

# 저장이 잘 됐는지 다시 읽어 확인 (read_csv 로 되불러오기)
saved = pd.read_csv("output/정제결과.csv")
print("다시 읽은 표 크기:", saved.shape)
print(saved.head(3))

---
## 이번 강의 정리

| 하고 싶은 일 | 도구 |
|---|---|
| 결측 찾기·세기 | `isna()`, `isna().sum()`, `notna()` |
| 결측 지우기·채우기 | `dropna(subset=...)`, `fillna(값)` |
| 이상치 감지·처리 | `describe()`, 조건 필터, `clip()`, `quantile()` |
| 자료형 변환 | `astype()`, `str.replace()`, `to_numeric()`, `to_datetime()` |
| 문자열 다듬기 | `.str.strip / replace / contains / lower / len / split` |
| 파생 변수 | `df['새열'] = ...`, `apply()`, `map()`, `np.where()` |

이제 여러분은 **지저분한 원본을 분석 가능한 깨끗한 표로** 바꿀 수 있습니다.

## ⏭️ 예고 — 다음 단원: EDA·시각화
깨끗한 데이터가 준비됐으니, 다음 단원에서는 본격적인 **탐색적 데이터 분석(EDA)** 으로 나아갑니다.
- **그룹별 집계**(`groupby`) — 카테고리별·요일별 매출 요약
- **표 병합**(`merge`)·**피벗**(`pivot_table`) — 여러 표를 합치고 재구성
- **시각화**(`matplotlib`·`seaborn`) — 막대·꺾은선·히스토그램으로 한눈에

오늘 만든 정제·파생 기술이 그 모든 분석의 **토대**가 됩니다. 수고하셨습니다!